In [8]:
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent
from typing import Any
from langgraph.runtime import Runtime
from langchain.agents import AgentState


@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <-----"
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <-----"
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <-----"
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <-----"
    return None


In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 通过load_dotenv()将.env中的变量加载为环境变量
# override=True表示：无论你当前的操作系统、终端或者虚拟环境中是否已经存在同名的环境变量，
# 都会强行用 .env 文件里写的值去覆盖它
load_dotenv(override=True)


# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


# 创建LLM
model = init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek:deepseek-flash" # 冒号前面直定模型厂商，冒号后面指定模型
)

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ]
)


response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

你好！你这条消息看起来像是被截断了（只有几个箭头和 hook 名字），我先猜一下你的意思：你大概想问 **`before_agent` 和 `before_model` 这两个生命周期钩子的区别与触发顺序**。

如果是这个意思，通用答案是这样的：

```
用户输入 "你好"
      │
      ▼
  before_agent        ← 整个 agent 一次运行只触发 1 次
      │
      ▼
 ┌── 循环 ────────────────────────────┐
 │  before_model      ← 每次调 LLM 前都触发（可能 N 次）│
 │        │                                            │
 │        ▼                                            │
 │     LLM 推理 → 有 tool_call ? → 执行工具 → 回到 before_model │
 └────────────────────────────────────┘
      │
      ▼
  after_agent / 返回结果
```

**关键区别**

| | `before_agent` | `before_model` |
|---|---|---|
| 作用域 | 外层，整个 agent run | 内层，单次模型调用 |
| 触发次数 | 1 次 | 1~N 次（工具循环里反复触发） |
| 典型用途 | 鉴权、开 session、注入用户上下文、改写输入、选模型 | 裁剪/压缩历史、注入 system prompt、计 token、加护栏、埋点 |

一句话记：**`bef

In [9]:
from langchain.agents.middleware import AgentMiddleware

class MyMiddleware(AgentMiddleware):
    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <-----"
        return None

    def after_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_model <-----"
        return None

    def before_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_agent <-----"
        return None

    def after_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_agent <-----"
        return None

In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    middleware=[
        MyMiddleware(),
    ]
)


response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

你好！你画的两个钩子 —— `before_agent` 和 `before_model` —— 是 Agent 中间件（middleware）里最常见的两个生命周期回调。它们的核心区别是**作用范围**和**触发频率**：

## 执行结构

```
before_agent        ← 整次 agent 调用只执行 1 次
  └─ 循环 (agent loop)
       before_model ← 每次调用 LLM 前都执行
         → LLM 调用
       after_model
         → 工具调用 / 决策
       ...
  after_agent       ← 结束时执行 1 次
```

## 对比

| | `before_agent` | `before_model` |
|---|---|---|
| 触发次数 | 每次 agent 调用 **1 次** | 每一轮 LLM 请求 **1 次**（可能在循环中多次） |
| 作用对象 | 整个 agent 会话 / 状态 | 单次模型请求的输入消息 |
| 典型用途 | 初始化状态、加载记忆、鉴权、设置全局配置、注入静态系统提示 | 裁剪/摘要历史消息、动态注入上下文（RAG、时间、用户信息）、token 预算控制、guardrails、按轮修改 prompt |
| 能否影响循环 | 一般只做前置准备 | 直接决定这一轮模型看到什么 |

## 常见用法示例

**`before_agent`** —— 每次会话开始做一次的事：

```python
def before_agent(state, runtime):
    state["user"] = load_user(runtime.context.user_

wrap_model_call

In [13]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "---> wrap_model_call_after <---"

    return response


agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！👋

我看到你的消息里带着 `wrap_model_call_before` 这样的标记，看起来像是某种框架（比如 LangGraph/LangChain 中间件）的钩子名，或者是一次接口测试。不过我这边没有对应的上下文——我不是某个具体框架的运行组件，也不会因为看到这类标记就切换到某种特殊模式。

如果你是想：
- **测试连通性/格式**：收到了，正常回复中 ✅
- **调试 middleware 钩子**：可以把代码或报错贴出来，我帮你看
- **问别的问题**：直接说就行

需要我做什么？---> wrap_model_call_after <---


wrap_tool_call

In [14]:
from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_00_Y03Xw2RBgvqeiJ7owmBt2836'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_Y03Xw2RBgvqeiJ7owmBt2836'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_Y03Xw2RBgvqeiJ7owmBt2836)
 Call ID: call_00_Y03Xw2RBgvqeiJ7owmBt2836
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

北京今天的天气不错哦～ 顺便一提，明天北京的天气也很好。出门可以放心安排活动，不过还是建议出门前再看一眼实时天气，注意适当增减衣物。
